# Applied Intelligent Systems - Lecture 7: Agentic AI in Data Science

In the previous lectures, we have explored the fundamentals of Language Models (LMs) and their applications for summarizing text, generating code, and answering questions. However, LMs so far are **passive tools** that require users to provide specific prompts to get the desired output. 

Agentic AI transforms LMs from passive, reactive knowledge systems into active systems capable of autonomously performing tasks, making decisions, and interacting with their environment. 
Instead of just answering questions, an AI Agent can plan and act to achieve specific goals, using tools and resources at its disposal.
An agent will break the task into smaller steps, decide which tools to use, and iteratively refine its approach based on feedback and results until the task is completed.

The transition from passive LMs to Agentic AI provides benefits, but also introduces new challenges and significant complexity compared to standard LM pipelines.

Our starting point is a standalone LM that can only generate text based on a given prompt. We will then enhance this LM with tool-use capabilities, allowing it to interact with external resources and perform more complex tasks autonomously.

After this lecture, you will be able to:
- Understand the concept of Agentic AI and how it differs from standard LMs.
- Implement a simple Agentic AI using the smolagents library and the Hugging Face Inference API.
- Add tool-use capabilities to your agent, enabling it to interact with external resources and perform complex tasks autonomously.
- Define and implement a custom tool for your agent to use in its decision-making process.

Let's get started!

---

## Variant 1: Standalone LM using the Hugging Face Inference API

As model, we will use the **Qwen 2.5-7B Instruct** model, which is a small but powerful language model that can generate high-quality text based on user prompts.

In [ ]:
import dotenv

# Some globals first
HF_ACCESS_TOKEN = dotenv.get_key(".env", "HF_ACCESS_TOKEN")
MODEL = "Qwen/Qwen2.5-7B-Instruct"
USER_PROMPT = "Which wild bee species can I likely observe today in Kufstein?"

We will start with a simple implementation of a standalone LM using the Hugging Face Inference API:

In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(api_key=HF_ACCESS_TOKEN)

In [ ]:
response = client.chat.completions.create(
    model=MODEL, 
    messages=[{
        "role": "user", 
        "content": USER_PROMPT
    }],
    max_tokens=500
)

In [ ]:
print(response.choices[0].message.content)

As expected, the model attempts to answer the question based on its training data, but it does not have access to real-time information about the current date or location, so it cannot provide a more accurate answer to the user's question.

Next, we will enhance this LM with tool-use capabilities, allowing it to interact with external resources and perform more complex tasks autonomously.

## Variant 2: Agentic AI using the `smolagents` library

As the next step, we will implement a simple Agent capabale of performing a task autonomously. The agent will have access to a set of tools and a LM as its engine.

To incorporate tool-use capabilities, we will exchange the simple HF inference client with the `smolagents` library, which provides a framework for building agents that can interact with tools and resources.

The two main ways to build an agent in `smolagents` are Tool Calling Agents and Coding Agents `CodeAgent`.

A **Tool Calling Agent** is the "standard" approach, where the agent selects a tool and provides inputs in a structured format (e.g., JSON). This approach for building agents is reliable for simple tasks but less flexible for complex reasoning.

A **Coding Agent** is a more flexible approach, where the agent generates code to solve the task. This allows for more complex reasoning and decision-making, but it also requires the agent to be able to write correct code and handle errors effectively.

To build an agent, we need at least two elements:

- `tools`: a list of tools the agent has access to.
- `model`: an LLM that serves as the engine of the agent.

Tools can be downloaded from the Hugging Face Hub or other frameworks. We can also create **custom tools** by writing our own functions. This will be explained in a later section.

For the **model**, the framework provides distinct classes to connect to different LLM providers:

- `InferenceClientModel` is the default class, which connects to Hugging Face's serverless inference service, allowing developers to run models hosted on the Hugging Face Hub. I.e., the agent runs on Hugging Face infrastructure, and no local GPU is required. However, only a small number of free tokens are offered per month for experimentation and prototyping. Beyond that quota, usage becomes pay-as-you-go under provider rates.
- `HfApiModel` class also uses Hugging Face's free inference API to give access to open-source LLMs and other models without local hosting. Still, availability and cost depend on the compute requirements of the chosen model, and for medium to large LLMs or frequent use, you may quickly exhaust free credits and need to pay for additional usage. The `HfAPIModel` interface is older legacy method for running inference on Hugging Face from older `huggingface_hub` versions, while `InferenceClientModel` is the modern, fully supported API that provides consistent outputs, streaming, and model support.
- `LiteLLMModel` class allows users to choose from a list of 100+ proprietary LLM providers, like OpenAI, Anthropic, or Azure. Agents can be powered by models like GPT-4o or Claude 4.5 Sonnet, which often have superior performance for highly complex tasks. Using proprietary LLMs requires providing the API key, and there are costs associated with using these models.

In [ ]:
from smolagents import ToolCallingAgent
from smolagents.models import InferenceClientModel

# Initialize the model
model = InferenceClientModel(
    model_id=MODEL, 
    api_key=HF_ACCESS_TOKEN
)

# Create an agent with the model, but no tools for now
agent = ToolCallingAgent(
    model=model, 
    tools=[]
)

# Run the agent with the user prompt
agent_response = agent.run(USER_PROMPT)

We can inspect the thinking process of the agent by calling `agent.replay()`, which shows the sequence of thoughts, actions, and tool calls made by the agent to arrive at its final answer. This is useful for debugging and understanding how the agent is reasoning through the problem.

In [ ]:
agent.replay()

Another way to inspect the agent's thinking process is to inspect the `agent.memory.steps`, which contains the raw sequence of thoughts, actions, and tool calls made by the agent during its reasoning process. This allows for a more detailed analysis of the agent's decision-making and can help identify any issues or areas for improvement in the agent's reasoning.

In [ ]:
agent.memory.steps

### Adding Custom Tools

So the LM realizes that it needs to gather current weather information and bee sightings to answer the user's question.
Let's see how we can tools to enable the agent to access real-time information and perform the necessary steps autonomously.

We can incorporate tools into our agent by providing a list of tools it can use. A simple example of a tool is `DuckDuckGo`, which allows the agent to perform web searches and retrieve real-time information from the internet.

Here, we will add custom tools:
- `get_current_date`: a tool that retrieves the current date.
- `get_weather`: a tool that retrieves the current weather information for a given location and date.
- `get_gbif_taxon_occurrences`: a tool that retrieves occurrence data for given taxa from a biodiversity database (GBIF).

The `smolagents` library allows creating custom tools using two main approaches:

1. Using the `@tool` decorator for simple function-based tools.
2. Creating a subclass of `Tool` for more complex functionality.

The `@tool` decorator is the recommended way to define simple tools. It simply requires writing a standard Python function with
- type hints, 
- a correctly formatted docstring, 
- and the `@tool` decorator. 

The framework parses this function to automatically generate the tool definition. This approach is much simpler than other frameworks that require complex class inheritance or JSON schema definitions to create custom tools.

In [ ]:
from smolagents import tool

from datetime import date
import requests
import os
import pandas as pd


@tool
def get_current_date() -> str:
    """Returns the current date in ISO format (YYYY-MM-DD)."""
    return date.today().isoformat()


@tool
def get_weather(lat: float, lon: float, date: str = None) -> dict:
    """
    Fetches weather data for a location and date. If date is None, returns current weather.

    Args:
        lat: Latitude of the location (Kufstein: 47.58)
        lon:  Longitude of the location (Kufstein: 12.17)
        date: Optional date for historical weather (YYYY-MM-DD)
    
    Returns:
        A dictionary with weather information or an error message.
    """
    if date:
        # OpenMeteo Historical
        url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={date}&end_date={date}&daily=temperature_2m_max,precipitation_sum,windspeed_10m_max&timezone=auto"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return {
                "date": date,
                "temp_max": data['daily']['temperature_2m_max'][0],
                "rain": data['daily']['precipitation_sum'][0],
                "wind_max": data['daily']['windspeed_10m_max'][0],
                "source": "OpenMeteo Historical"
            }
    else:
        # OpenMeteo Current
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return {
                "date": get_current_date(),
                "temp": data['current_weather']['temperature'],
                "wind": data['current_weather']['windspeed'],
                "condition_code": data['current_weather']['weathercode'],
                "source": "OpenMeteo Current"
            }
    return "Error: Could not retrieve weather data."


@tool
def get_gbif_taxon_occurrences(lat: float, lon: float, taxon_keys: list = [], radius_km: int = 10, month: int = None) -> pd.DataFrame:
    """
    Retrieves occurrence records from GBIF for specified taxon keys within a radius of a location, optionally filtered by month.

    Args:
        lat (float): Latitude of the center point for the search (Kufstein: 47.58) 
        lon (float): Longitude of the center point for the search (Kufstein: 12.17)
        taxon_keys (list): List of GBIF taxon keys (int) for wild bee families
        radius_km (int): Search radius in kilometers
        month (int): Optional filter for month of observation (1-12)

    Returns:
        A pandas DataFrame with columns: species, date, lat, lon, or an error message if the API call fails.
    """

    # load from file if available
    gbif_occurrences_file = os.path.join("data", "gbif_sightings.csv")
    
    if os.path.exists(gbif_occurrences_file):
        df = pd.read_csv(gbif_occurrences_file)
        print("Loaded GBIF occurrences to DataFrame:")
        print(df.head())
        return df
            

    url = f"https://api.gbif.org/v1/occurrence/search?geoDistance={lat},{lon},{radius_km}km&limit=300"
    
    for key in taxon_keys:
        url += f"&taxonKey={key}"

    if month:
        url += f"&month={month}"
    
    response = requests.get(url)
    if not response.status_code == 200:
        return f"Error: {response.status_code} - {response.text}"

    results = response.json().get('results', [])
    
    sightings = []
    for record in results:
        sightings.append({
            "species": record.get('species', 'Unknown'),
            "date": record.get('eventDate', 'Unknown'),
            "lat": record.get('decimalLatitude', 'Unknown'),
            "lon": record.get('decimalLongitude', 'Unknown')
        })

    df = pd.DataFrame(sightings)
    print("Fetched GBIF occurrences and created DataFrame:")
    print(df.head())
    return df


@tool
def get_bee_taxon_keys() -> list:
    """
    Returns a list of GBIF taxon keys for seven wild bee families, i.e., 
    Apidae, Andrenidae, Colletidae, Halictidae, Megachilidae, Melittidae, Stenotritidae.
    """
    # These are the GBIF taxon keys for the respective families
    return [7901, 4334, 7905, 7908, 7911, 4345, 7916]

Now we can add these tools to our agent:

In [ ]:
# Create an agent with the model, this time with our custom tools
agent = ToolCallingAgent(
    model=model, 
    tools=[ # add our tools to the agent
        get_current_date, 
        get_weather, 
        get_gbif_taxon_occurrences,
        get_bee_taxon_keys,
    ],
)

# Run the agent with the user prompt
agent_response = agent.run(USER_PROMPT)

print(agent_response)

In [ ]:
agent.replay()

In [ ]:
for step in agent.memory.steps[1:]:

    print(f"\n\n--- Step {getattr(step, 'step_number')} ---")

    for attr in ["code_action", "action_output", "tool_calls", "observations",]:
        print(f"\n{attr}")
        print(getattr(step, attr))


That is a more grounded reply, but as a Data Scientist, we want to have a more detailed and sophisticated answer. 

We can enhance our Agent's capabilities by writing and executing code in order to analyze the data it retrieves from the tools. 
This will allow the agent to not only gather information but also to process and interpret it to provide a more comprehensive answer to the user's question.

### Coding Agents

Differently from the `ToolCallingAgent` in `smolagents` that generates tool calls as JSON structures, a `CodeAgent` writes and executes Python code blocks. I.e., the `CodeAgent` writes a Python script that calls tools. This makes the agent more capable because it can perform math operations, process lists, and use logic (like if statements) within a single step.

In the following example, we create a new `InferenceClientModel` using a different LLM that is more suitable for Code Agents, `Qwen2.5-Coder-32B-Instruct`. Next, we define a new agent using `CodeAgent` and run the agent.

**Security of Code Agents**

Allowing AI agents to generate and execute Python code also introduces security risks, since a compromised model could take control of the host environment, run harmful commands, or access sensitive data.

- To address this, `smolagents` suggests, and in many production cases requires, *sandboxed execution* as the default and safest strategy. Production deployments typically use E2B, a cloud-based isolated environment that prevents any generated code from using the host machine.
- Developers who need local control can run agents inside *Docker containers*, where the AI agents work in an isolated environment to limit potential damage.
- In addition, `smolagents` allows developers to *restrict imports* using the `additional_authorized_imports` argument. By limiting the agent to safe libraries like `math`, `pandas`, `json`, etc., developers can significantly reduce the attack risks, though this does not provide the full isolation offered by a sandbox.

In [ ]:
from smolagents import CodeAgent

# Create a client model using the Qwen2.5-Coder-32B-Instruct model
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct", 
    token=HF_ACCESS_TOKEN
)

# This is our CodeAgent with our tools added
agent = CodeAgent(
    tools=[
        get_current_date, 
        get_weather, 
        get_gbif_taxon_occurrences,
        get_bee_taxon_keys
    ],
    add_base_tools=True,
    additional_authorized_imports=["numpy", "pandas"], # allow the agent to import numpy and pandas in its code
    model=model,
    max_steps=5 # limit the number of steps to prevent infinite loops during testing
)

The argument `add_base_tools=True` automatically includes default tools, such as Python Interpreter Tool, Final Answer Tool, etc., in addition to our custom tools.

In [ ]:
# Run the agent
agent_response = agent.run(USER_PROMPT)

print(agent_response)

I'm not satisfied with this answer.
Let's improve the prompt to encourage the agent to create a more detailed plan, use the tools effectively, and **model** the data it retrieves to provide a more comprehensive answer.

In [ ]:
enhanced_prompt = """
You are an expert Entomologist and Data Scientist specializing in Tyrolean biodiversity.
Your goal is to return the likelihood of observing a wild bee species in Kufstein based on current date, weather conditions, and historical sighting data along with corresponding weather data.

Get the current date and weather conditions for Kufstein using the provided tools.
Get a list of wild bee families, then get their historical sightings in the current month from GBIF for the area around Kufstein (radius of 10km).
Get the weather data for the days of the sightings.
Then analyze the relation between sightings and weather conditions using the provided Random Forest modeling tool.

Start by getting an overview on the tools available to you.
Then create a detailed plan on how to use them to solve the problem.
Where possible, use the provided tools to load pre-fetched data from files to save time instead of making API calls.
There is no need to edit, convert, or filter the data, just use the tools to retrieve the necessary information and then apply the modeling tool to analyze it.

Kufstein Location: lat=47.58, lon=12.17

Final Answer: [List of species with their probabilities]
"""

enhanced_prompt += "Original user question: " + USER_PROMPT

Also, let us help the agent by providing two other tools
- for loading prefetched weather data: `get_prefetched_weather_data()`
- one for running a statistical model: `run_statistical_model()`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

@tool
def get_prefetched_weather_data() -> pd.DataFrame:
    """
    Fetches prefetched weather data for all the dates of the GBIF sightings from a local file.
    """
    weather_data_file = os.path.join("data", "weather_data.csv")
    if os.path.exists(weather_data_file):
        return pd.read_csv(weather_data_file)
    else:
        return "Error: Prefetched weather data file not found."


@tool
def analyze_bee_sightings_with_weather(
    bee_sightings_df: pd.DataFrame, 
    weather_df: pd.DataFrame, 
    current_temp: float,
    current_rain: float,
    current_wind: float,
    current_date: str,
    ) -> dict:
    """
    Fits a Random Forest Classifier to the prefetched bee sightings and weather data to model the likelihood of observing each bee species based on weather conditions and date.    
    
    Args:
        bee_sightings_df: DataFrame with columns 'species', 'date', 'lat', 'lon' for bee sightings returned by `get_gbif_taxon_occurrences`
        weather_df: DataFrame with columns 'date', 'temp_max', 'rain_sum', 'wind_max' for weather data returned by `get_prefetched_weather_data`
        current_temp: Current temperature
        current_rain: Current rain
        current_wind: Current wind
        current_date: Current date in ISO format (YYYY-MM-DD)
    """
    # equalize the date formats, ignore the time component if present
    bee_sightings_df['date'] = pd.to_datetime(bee_sightings_df['date'], errors='coerce', format="ISO8601").dt.date
    weather_df['date'] = pd.to_datetime(weather_df['date'], errors='coerce').dt.date
    merged_df = pd.merge(bee_sightings_df, weather_df, on="date")

    # convert date to day of year
    merged_df['doy'] = pd.to_datetime(merged_df['date']).dt.dayofyear

    # unique bee species and their counts
    species_list = merged_df['species'].value_counts()

    # define features and targets for modeling
    features = ['temp_max', 'rain_sum', 'wind_max', 'doy']
    x_data = merged_df[features]
    y = merged_df['species']

    # categorical encoding of the species names (=the target variable)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    # Fit a Random Forest Classifier
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(x_data, y_encoded)

    # predict probabilities using today's data
    doy = pd.to_datetime(current_date).dayofyear
    input_data = pd.DataFrame(
        [[current_temp, current_rain, current_wind, doy]], 
        columns=features
    )
    
    # Get probabilities for every species
    probs = rf.predict_proba(input_data)[0]
    
    # Combine with species names
    result = dict(zip(le.classes_, probs))
    
    # Sort by highest likelihood
    prediction = dict(sorted(result.items(), key=lambda item: item[1], reverse=True))

    print("Species Likelihoods:")
    for species, score in prediction.items():
        if score > 0: # Only show species with a chance
            print(f"{species}: {score:.1%}")
    return prediction

In [ ]:
# Define a new CodeAgent with the enhanced tools
agent = CodeAgent(
    tools=[
        get_current_date, 
        get_weather, 
        get_gbif_taxon_occurrences,
        get_bee_taxon_keys,
        get_prefetched_weather_data, # we add this as a shortcut to load the weather data for the sightings without making API calls 
        analyze_bee_sightings_with_weather, # this is our modeling based on Random Forests
    ],
    add_base_tools=True,
    # additional_authorized_imports=["numpy", "pandas"],
    model=model
)

# Run the agent using the enhanced prompt
agent_response = agent.run(enhanced_prompt)

In [ ]:
agent.replay()